# BirdCLEF 2026 — Distilled SED (Colab Pro+ 版)

**変更点 (vs Kaggle版):**
- Perch 埋め込みを学習前に一括プリ計算 → 学習ループ中の ONNX 推論を排除
- 学習が GPU バウンドになり 5〜10x 高速化
- Google Drive にデータ・出力を永続化

**初回セットアップ手順:**
1. Google Drive の `MyDrive/birdclef2026/` に `kaggle.json` をアップロード
2. 「ランタイム → すべてのセルを実行」

## 🧪 PSEUDO-LABEL Retraining Mode

**変更点 (vs original)**:
- `FOLDS = [0, 1, 2]` (Phase 3 Session 1 / Session 2 は [3, 4])
- S2 cellで `pseudo_labels.csv` 自動ロード → Y_SC hybrid化
- 実ラベル優先、unlabeled chunkにpseudoを補完 (max_prob > 0.30)

**事前準備**:
1. Kaggle上で `birdclef2026-pseudo-labels-generator` 実行完了確認
2. 下のセルで `pseudo_labels.csv` をColabにダウンロード
3. 通常通り Cell 1〜 を実行

**期待効果**: B1 fold 0 OOF 0.71 → 0.85+ 改善見込み, LB +0.005〜+0.010

In [ ]:
# =================================================================
# PSEUDO-LABEL DOWNLOAD (Colab)
# =================================================================
# Colab Secrets: KAGGLE_USERNAME + KAGGLE_KEY を使用
import os, json, requests
from pathlib import Path

PSEUDO_CSV = Path("./pseudo_labels.csv")

if PSEUDO_CSV.exists():
    print(f"✅ Already downloaded: {PSEUDO_CSV} ({PSEUDO_CSV.stat().st_size/1024:.1f} KB)")
else:
    # Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        KAGGLE_USER = userdata.get('KAGGLE_USERNAME').strip()
        KAGGLE_KEY  = userdata.get('KAGGLE_KEY').strip()
    except Exception as e:
        # Fallback: /root/.kaggle/kaggle.json (set by Cell 3 COLAB SETUP if already run)
        kj = Path('/root/.kaggle/kaggle.json')
        if kj.exists():
            creds = json.loads(kj.read_text())
            KAGGLE_USER = creds['username'].strip()
            KAGGLE_KEY  = creds['key'].strip()
        else:
            raise RuntimeError(
                f'Kaggle credentials not found.\n'
                f'  Set Colab secrets: KAGGLE_USERNAME + KAGGLE_KEY  OR  run Cell 3 first.\n'
                f'  Original error: {e}'
            )

    headers = {"Authorization": f"Bearer {KAGGLE_KEY}"}
    list_url = f"https://www.kaggle.com/api/v1/kernels/output?userName={KAGGLE_USER}&kernelSlug=birdclef2026-pseudo-labels-generator"

    print(f"Listing kernel outputs for {KAGGLE_USER}/birdclef2026-pseudo-labels-generator ...")
    r = requests.get(list_url, headers=headers, timeout=60)
    print(f"List status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        found = False
        for f in data.get('files', []):
            if 'pseudo_labels.csv' in f.get('fileName', '') or 'pseudo_labels.csv' in f.get('name', ''):
                dl_url = f.get('url')
                size = f.get('size') or f.get('fileSize', '?')
                print(f"Downloading pseudo_labels.csv ({size} bytes) ...")
                resp = requests.get(dl_url, headers=headers, timeout=300)
                if resp.status_code == 200:
                    PSEUDO_CSV.write_bytes(resp.content)
                    print(f"✅ Saved: {PSEUDO_CSV} ({PSEUDO_CSV.stat().st_size/1024:.1f} KB)")
                    found = True
                else:
                    print(f"❌ Download failed: {resp.status_code} {resp.text[:200]}")
                break
        if not found:
            print("⚠️ pseudo_labels.csv not found in kernel output. Available files:")
            for f in data.get('files', []):
                print(f"   - {f.get('fileName') or f.get('name')}")
    else:
        print(f"❌ List failed: {r.text[:300]}")
        print("\n⚠️ Fallback: manually download from")
        print("   https://www.kaggle.com/code/gorubachohu/birdclef2026-pseudo-labels-generator")
        print("   → Output tab → pseudo_labels.csv → Download")
        print("   → Upload to Colab via Files panel as ./pseudo_labels.csv")

In [ ]:
# =================================================================
# COLAB SETUP — Secrets 認証のみ
# =================================================================
import sys, os

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_COLAB:
    from google.colab import userdata

    DATA_DIR = '/content/data'
    OUT_DIR  = '/content/working'
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(OUT_DIR,  exist_ok=True)

    os.system('pip install -q kaggle')
    os.makedirs('/root/.kaggle', exist_ok=True)
    kaggle_key = userdata.get('KAGGLE_KEY').strip()
    kaggle_json = f'{{"username":"gorubachohu","key":"{kaggle_key}"}}'
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(kaggle_json)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle auth: OK')
    print(f'DATA_DIR : {DATA_DIR}')
    print(f'OUT_DIR  : {OUT_DIR}')

In [ ]:
# =================================================================
# DL-1: birdclef-2026 コンペデータ  (requests直接ダウンロード)
# kaggle CLI は KGAT_ トークンの Bearer 認証に失敗するため requests を直接使用
# =================================================================
import os, time, shutil, json as _json, subprocess
import requests as _req

DATA_DIR   = '/content/data'
dst        = f'{DATA_DIR}/birdclef-2026'
check_file = f'{dst}/sample_submission.csv'
zip_path   = f'{DATA_DIR}/birdclef-2026.zip'

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(check_file):
    print(f'already exists: {dst}')
else:
    if os.path.exists(dst):      shutil.rmtree(dst)
    if os.path.exists(zip_path): os.remove(zip_path)

    kaggle_json_path = '/root/.kaggle/kaggle.json'
    if not os.path.exists(kaggle_json_path):
        raise RuntimeError('kaggle.json not found → cell-setup を先に実行してください')

    with open(kaggle_json_path) as _f:
        _creds = _json.load(_f)
    _key  = _creds['key'].strip()
    _user = _creds['username'].strip()
    print(f'Kaggle auth: username={_user}  key={_key[:8]}...')

    _headers = {'Authorization': f'Bearer {_key}'}
    _url = 'https://www.kaggle.com/api/v1/competitions/data/download-all/birdclef-2026'

    print('Downloading birdclef-2026 ...')
    t0 = time.time()

    _r = _req.get(_url, headers=_headers, stream=True, allow_redirects=True, timeout=600)
    print(f'HTTP {_r.status_code}')
    if _r.status_code == 401:
        raise RuntimeError('401 Unauthorized → KAGGLE_KEY が無効です。Kaggle Settings でトークンを再発行してください')
    elif _r.status_code == 403:
        raise RuntimeError('403 Forbidden → https://www.kaggle.com/competitions/birdclef-2026/rules でルール承認が必要です')
    elif _r.status_code != 200:
        raise RuntimeError(f'Download failed: HTTP {_r.status_code}: {_r.text[:200]}')

    total = int(_r.headers.get('content-length', 0))
    downloaded = 0
    with open(zip_path, 'wb') as _f:
        for _chunk in _r.iter_content(chunk_size=64 * 1024 * 1024):  # 64MB chunks
            _f.write(_chunk)
            downloaded += len(_chunk)
            if total:
                print(f'\r  {downloaded/1e9:.1f}/{total/1e9:.1f} GB  ({downloaded/total*100:.0f}%)', end='', flush=True)
            else:
                print(f'\r  {downloaded/1e9:.1f} GB', end='', flush=True)
    print(f'\nzip: {downloaded/1e9:.1f} GB  ({time.time()-t0:.0f}s elapsed)')

    print('Unzipping ...')
    r2 = subprocess.run(
        f'unzip -q {zip_path} -d {dst}/ && rm -f {zip_path}',
        shell=True, capture_output=True, text=True)
    if r2.returncode != 0:
        print('unzip stderr:', r2.stderr[:300])

    elapsed = time.time() - t0
    if os.path.exists(check_file):
        subprocess.run(f'du -sh {dst}', shell=True)
        print(f'Done ({elapsed/60:.1f} min)')
    else:
        print(f'ERROR: {check_file} not found after unzip!')
        if os.path.exists(dst):
            contents = os.listdir(dst)
            print('Top-level:', contents[:20])
            for name in contents[:5]:
                sub = f'{dst}/{name}'
                if os.path.isdir(sub): print(f'  {name}/:', os.listdir(sub)[:10])

In [ ]:
# =================================================================
# DL-4: perch ONNX (Google Drive) + B1 weights (HuggingFace via S3)
# =================================================================
import os, zipfile

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# --- Perch ONNX: Google Drive から取得 ---
PERCH_ONNX = f'{DATA_DIR}/perch-onnx/perch_v2_no_dft.onnx'
DRIVE_ZIP   = '/content/drive/MyDrive/perch-v2-no-dft-onnx.zip'

if os.path.exists(PERCH_ONNX):
    print(f'Perch ONNX already exists: {PERCH_ONNX}')
else:
    os.makedirs(f'{DATA_DIR}/perch-onnx', exist_ok=True)

    # Drive マウント
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    if not os.path.exists(DRIVE_ZIP):
        raise FileNotFoundError(
            f'Drive に zip が見つかりません: {DRIVE_ZIP}\n'
            f'Kaggle から perch-v2-no-dft-onnx.zip をダウンロードして '
            f'Google Drive のマイドライブ直下に置いてください')

    print(f'Extracting Perch ONNX from Drive zip ...')
    with zipfile.ZipFile(DRIVE_ZIP) as zf:
        zf.extract('perch_v2_no_dft.onnx', f'{DATA_DIR}/perch-onnx/')
    print(f'  Done: {PERCH_ONNX}')

# --- B1 weights: S3 セル (load_backbone_weights_offline) で HuggingFace から自動取得 ---
print('B1 weights → handled by S3 cell via HuggingFace.')
print('All downloads complete!')

In [ ]:
# packages
if IS_COLAB:
    import subprocess, os
    # /tmp のキャッシュを使わない（tmpfs が小さいため）
    subprocess.run('pip install -q --no-cache-dir onnxruntime-gpu timm torchaudio onnx safetensors onnxscript', shell=True)
    # torchinductor キャッシュを /content に移動
    os.environ['TORCHINDUCTOR_CACHE_DIR'] = '/content/torchinductor_cache'
    os.makedirs('/content/torchinductor_cache', exist_ok=True)
else:
    # Kaggle: onnxruntime wheel is bundled
    pass

## S1 — Imports & Config

In [ ]:
# =================================================================
# S1 -- IMPORTS + CONFIG
# =================================================================
import os, sys, time, json, pickle, gc, random, math
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
from scipy.special import expit as sigmoid_np
import warnings
warnings.filterwarnings('ignore')

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    cc = torch.cuda.get_device_capability(0)
    print(f'CC:      sm_{cc[0]}{cc[1]}')

device = torch.device('cuda')
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
if IS_COLAB:
    DATA_DIR        = '/content/data'
    COMP_DIR        = Path(f'{DATA_DIR}/birdclef-2026')
    FOCAL_AUDIO_DIR = COMP_DIR / 'train_audio'
    SC_AUDIO_DIR    = COMP_DIR / 'train_soundscapes'
    PERCH_ONNX_PATH = Path(f'{DATA_DIR}/perch-onnx/perch_v2_no_dft.onnx')
    B1_WEIGHTS_PATH = f'{DATA_DIR}/b1-weights/model.safetensors'
    OUT_DIR         = Path('/content/working')
else:  # Kaggle
    COMP_DIR        = Path('/kaggle/input/competitions/birdclef-2026')
    FOCAL_AUDIO_DIR = COMP_DIR / 'train_audio'
    SC_AUDIO_DIR    = COMP_DIR / 'train_soundscapes'
    PERCH_ONNX_PATH = Path('/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx')
    B1_WEIGHTS_PATH = '/kaggle/input/timm-efficientnet-b1-ns-jft-in1k/model.safetensors'
    OUT_DIR         = Path('/kaggle/working')

OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_PATH     = COMP_DIR / 'train_soundscapes_labels.csv'
TAXONOMY_PATH   = COMP_DIR / 'taxonomy.csv'
SAMPLE_SUB_PATH = COMP_DIR / 'sample_submission.csv'
TEST_DIR        = COMP_DIR / 'test_soundscapes'

NUM_CLASSES = 234
SR          = 32000

TRAIN_DURATION = 5
VAL_DURATION   = 5
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION

N_FOLDS    = 5
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 256
FMIN       = 20
FMAX       = 16000

# ─── PSEUDO-LABEL CONFIG ────────────────────────────────────────────────
USE_PSEUDO_LABELS = True
PSEUDO_LABELS_PATH = Path("./pseudo_labels.csv")  # upload to Colab working dir
PSEUDO_CONFIDENCE_THRESHOLD = 0.30  # only use pseudo where max_prob > threshold
PSEUDO_LABEL_WEIGHT = 0.7  # downweight pseudo-label loss vs real labels (1.0)
# ─────────────────────────────────────────────────────────────────────────

BACKBONE_NAME     = 'tf_efficientnet_b1.ns_jft_in1k'
USE_PERCH_DISTILL = True
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0

MODE   = 'train'
DEBUG  = False
FOLDS  = [0, 1, 2]   # Phase 3 Session 1: all folds (0,1,2 first, then 3,4 in session 2)
EPOCHS = 6
BATCH  = 32
LR     = 5e-4
MIN_LR = 1e-5
WD     = 1e-4
WARMUP_EPOCHS = 2

MIN_SAMPLE = 20
AUG_PROB   = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

USE_FOCAL_MIXUP     = True
MIXUP_PROB          = 0.5
MIXUP_ALPHA         = 0.4
MIXUP_HARD          = True
USE_FOCAL_SC_MIXUP  = True
FOCAL_SC_MIXUP_PROB = 0.5
FOCAL_SC_MIXUP_ALPHA= 0.4

FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

USE_FOCAL           = True
USE_FOCAL_SECONDARY = True
USE_LABELED_SC      = True
SHARES = {'focal': 0.9, 'sc': 0.1}
SOURCE_WEIGHTS = {'focal': 1.0, 'focal_missing': 0.0, 'sc': 1.0}

EMBED_CACHE = {}

# ------------------------------------------------------------------
# Audio loading (torchaudio ベース、waveform cache 不要)
# ------------------------------------------------------------------
_AUDIO_CACHE = {}
MAX_AUDIO_CACHE = 600  # focal ~320KB × 600 ≈ 190MB

def load_audio_np(path):
    """任意の音声ファイルを 32kHz float32 numpy array で返す"""
    path = str(path)
    if path in _AUDIO_CACHE:
        return _AUDIO_CACHE[path]
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != SR:
            wav = torchaudio.functional.resample(wav, sr, SR)
        a = wav.squeeze(0).numpy().astype(np.float32)
    except Exception as e:
        print(f'  load_audio_np error: {path}: {e}')
        return None
    if len(_AUDIO_CACHE) >= MAX_AUDIO_CACHE:
        _AUDIO_CACHE.pop(next(iter(_AUDIO_CACHE)))
    _AUDIO_CACHE[path] = a
    return a

_focal_path_cache = {}
def resolve_focal_path(label, filename):
    key = (label, filename)
    if key in _focal_path_cache:
        return _focal_path_cache[key]
    p = FOCAL_AUDIO_DIR / label / filename
    if not p.exists():
        p = FOCAL_AUDIO_DIR / filename  # fallback: flat structure
    if not p.exists():
        p = None
    _focal_path_cache[key] = p
    return p

print(f'Backbone : {BACKBONE_NAME}')
print(f'Epochs   : {EPOCHS}  Batch: {BATCH}  Folds: {FOLDS}')
print(f'OUT_DIR  : {OUT_DIR}')
print(f'Audio    : torchaudio (waveform cache 不要)')

## S2 — Load Data

In [ ]:
# =================================================================
# S2 -- LOAD DATA (waveform cache 不要、コンペデータから直接構築)
# =================================================================
import onnxruntime as ort

sample_sub     = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX      = {l: i for i, l in enumerate(PRIMARY_LABELS)}
taxonomy       = pd.read_csv(TAXONOMY_PATH)
label_to_taxon = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name'].astype(str)))
TAXON_MASKS    = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS) if label_to_taxon.get(l,'') == t])
                  for t in ['Aves','Amphibia','Insecta','Mammalia','Reptilia']}

# ------------------------------------------------------------------
# Focal: train.csv から audio_cache_meta を構築
# ------------------------------------------------------------------
train_df = pd.read_csv(COMP_DIR / 'train.csv')
audio_cache_meta = train_df[train_df['primary_label'].isin(LABEL2IDX)].copy().reset_index(drop=True)
audio_cache_meta['original_idx'] = audio_cache_meta.index
# cache_file = "label/filename" （resolve_focal_path へのキー）
audio_cache_meta['cache_file'] = (audio_cache_meta['primary_label'].astype(str)
                                   + '/' + audio_cache_meta['filename'].astype(str))
print(f'Focal audio: {len(audio_cache_meta)} files')

# ------------------------------------------------------------------
# Soundscape: ラベルCSV + torchaudio.info で全 5 秒窓を構築
# ------------------------------------------------------------------
sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
sc_labels_raw['start_sec'] = pd.to_timedelta(sc_labels_raw['start']).dt.total_seconds().astype(int)

# (filename, start_sec) -> labels dict
_sc_label_dict = {}
for _, row in sc_labels_raw.iterrows():
    key = (row['filename'], int(row['start_sec']))
    _sc_label_dict.setdefault(key, []).append(str(row['primary_label']))

sc_filenames = sorted(sc_labels_raw['filename'].unique())
print(f'Soundscape files: {len(sc_filenames)}')

sc_windows = []
for sc_file in sc_filenames:
    path = SC_AUDIO_DIR / sc_file
    site = sc_file.split('_')[0]  # ファイル名先頭がサイトコード（例: S22_...）
    try:
        info = torchaudio.info(str(path))
        duration_sec = info.num_frames / info.sample_rate
    except Exception:
        duration_sec = 60  # 取得できなければ 60 秒とみなす
    n_windows = int(duration_sec // 5)
    for i in range(n_windows):
        start_sec = i * 5
        sc_windows.append({
            'filename': sc_file,
            'start_sec': start_sec,
            'site': site,
            'label_list': _sc_label_dict.get((sc_file, start_sec), []),
        })

sc_cache_meta = pd.DataFrame(sc_windows)
print(f'SC windows: {len(sc_cache_meta)}')

# Y_SC ラベル行列
Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    for lbl in row['label_list']:
        if lbl in LABEL2IDX:
            Y_SC[i, LABEL2IDX[lbl]] = 1.0
labeled_sc_mask = Y_SC.sum(axis=1) > 0
print(f'SC labels: {labeled_sc_mask.sum()}/{len(Y_SC)} labeled')

# ------------------------------------------------------------------
# Fold assignment
# ------------------------------------------------------------------
audio_for_split = audio_cache_meta.drop_duplicates('original_idx').reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split['fold'] = -1
for fold, (_, vi) in enumerate(skf.split(audio_for_split, audio_for_split['primary_label'])):
    audio_for_split.loc[vi, 'fold'] = fold
audio_cache_meta = audio_cache_meta.merge(
    audio_for_split[['original_idx','fold']], on='original_idx', how='left')

sc_files_df = sc_cache_meta[['filename','site']].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files_df['fold'] = -1
for fold, (_, vi) in enumerate(gkf.split(sc_files_df, groups=sc_files_df['filename'])):
    sc_files_df.loc[sc_files_df.index[vi], 'fold'] = fold
file_to_fold = dict(zip(sc_files_df['filename'], sc_files_df['fold']))
sc_cache_meta['fold'] = sc_cache_meta['filename'].map(file_to_fold).fillna(-1).astype(int)

# Upsample rare species
counts = audio_cache_meta['primary_label'].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index
extra = []
for sp in rare_species:
    rows = audio_cache_meta[audio_cache_meta['primary_label'] == sp]
    for _ in range(int(np.ceil(MIN_SAMPLE / len(rows))) - 1):
        extra.append(rows)
n_before = len(audio_cache_meta)
if extra:
    audio_cache_meta = pd.concat([audio_cache_meta] + extra, ignore_index=True)
print(f'Upsampled {len(rare_species)} rare species: {n_before} -> {len(audio_cache_meta)}')

non_s22_mask_sc = sc_cache_meta['site'].values != 'S22'
print(f'S22 mask: non-S22={non_s22_mask_sc.sum()}')

# ─────────────────────────────────────────────────────────────────────────
# PSEUDO-LABEL INTEGRATION
# ─────────────────────────────────────────────────────────────────────────
# Strategy:
#   1. Load pseudo_labels.csv (output of Kaggle pseudo-labels-generator)
#   2. Add chunks from pseudo-only files (no real labels) to sc_cache_meta
#   3. Build hybrid Y_SC: real labels override pseudo where both exist
#   4. Rebuild fold assignment via GroupKFold (file-level)
#
# is_pseudo_mask records which rows are pseudo-only — used for loss weighting.
# ─────────────────────────────────────────────────────────────────────────
is_pseudo_mask = np.zeros(len(sc_cache_meta), dtype=bool)
if USE_PSEUDO_LABELS and PSEUDO_LABELS_PATH.exists():
    print(f"\n[PSEUDO] Loading from {PSEUDO_LABELS_PATH}")
    pseudo_df = pd.read_csv(PSEUDO_LABELS_PATH)
    print(f"[PSEUDO] Loaded {len(pseudo_df)} rows × {len(pseudo_df.columns)-1} class cols")

    # Parse row_id "BC2026_Train_..._<end_sec>" → (filename, start_sec)
    def _parse_row_id(rid):
        parts = rid.rsplit("_", 1)
        return parts[0] + ".ogg", int(parts[1]) - 5  # start_sec = end_sec - 5

    pseudo_class_cols = [c for c in pseudo_df.columns if c != "row_id"]
    pseudo_lookup = {}
    for _, row in pseudo_df.iterrows():
        fn, ss = _parse_row_id(row["row_id"])
        pseudo_lookup[(fn, ss)] = row[pseudo_class_cols].to_numpy(np.float32)
    print(f"[PSEUDO] Lookup built: {len(pseudo_lookup)} (filename, start_sec) entries")

    # Identify files only present in pseudo-labels (not in real labels)
    pseudo_files = set(fn for fn, _ in pseudo_lookup.keys())
    existing_files = set(sc_cache_meta['filename'].unique())
    new_files = pseudo_files - existing_files
    print(f"[PSEUDO] Files: {len(existing_files)} real, {len(new_files)} pseudo-only")

    # Build new windows for pseudo-only files
    extra_windows = []
    for sc_file in sorted(new_files):
        path = SC_AUDIO_DIR / sc_file
        site = sc_file.split('_')[0]
        try:
            info = torchaudio.info(str(path))
            duration_sec = info.num_frames / info.sample_rate
        except Exception:
            duration_sec = 60
        n_windows = int(duration_sec // 5)
        for i in range(n_windows):
            start_sec = i * 5
            if (sc_file, start_sec) in pseudo_lookup:
                extra_windows.append({
                    'filename': sc_file,
                    'start_sec': start_sec,
                    'site': site,
                    'label_list': [],  # no real labels
                })

    if extra_windows:
        extra_df = pd.DataFrame(extra_windows)
        n_before = len(sc_cache_meta)
        sc_cache_meta = pd.concat([sc_cache_meta, extra_df], ignore_index=True)
        print(f"[PSEUDO] sc_cache_meta: {n_before} → {len(sc_cache_meta)} (+{len(extra_df)} pseudo-only chunks)")

        # Rebuild Y_SC with hybrid labels
        Y_SC_new = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
        is_pseudo_mask = np.zeros(len(sc_cache_meta), dtype=bool)
        n_real, n_pseudo, n_skip = 0, 0, 0
        for i, row in sc_cache_meta.iterrows():
            # Real label (priority)
            if row['label_list']:
                for lbl in row['label_list']:
                    if lbl in LABEL2IDX:
                        Y_SC_new[i, LABEL2IDX[lbl]] = 1.0
                n_real += 1
            else:
                # Try pseudo
                key = (row['filename'], int(row['start_sec']))
                if key in pseudo_lookup:
                    probs = pseudo_lookup[key]
                    if probs.max() > PSEUDO_CONFIDENCE_THRESHOLD:
                        Y_SC_new[i] = probs
                        is_pseudo_mask[i] = True
                        n_pseudo += 1
                    else:
                        n_skip += 1
                else:
                    n_skip += 1
        Y_SC = Y_SC_new
        labeled_sc_mask = Y_SC.sum(axis=1) > 0
        print(f"[PSEUDO] Y_SC: {n_real} real, {n_pseudo} pseudo (>{PSEUDO_CONFIDENCE_THRESHOLD}), "
              f"{n_skip} skipped | total labeled {labeled_sc_mask.sum()}/{len(Y_SC)}")

        # Re-do fold assignment (GroupKFold by filename) on expanded set
        sc_files_df = sc_cache_meta[['filename','site']].drop_duplicates().reset_index(drop=True)
        gkf = GroupKFold(n_splits=N_FOLDS)
        sc_files_df['fold'] = -1
        for fold, (_, vi) in enumerate(gkf.split(sc_files_df, groups=sc_files_df['filename'])):
            sc_files_df.loc[sc_files_df.index[vi], 'fold'] = fold
        file_to_fold = dict(zip(sc_files_df['filename'], sc_files_df['fold']))
        sc_cache_meta['fold'] = sc_cache_meta['filename'].map(file_to_fold).fillna(-1).astype(int)
        non_s22_mask_sc = sc_cache_meta['site'].values != 'S22'
        print(f"[PSEUDO] Refreshed folds + non_s22 mask: {non_s22_mask_sc.sum()}")
else:
    if USE_PSEUDO_LABELS:
        print(f"[PSEUDO] WARNING: USE_PSEUDO_LABELS=True but file not found: {PSEUDO_LABELS_PATH}")
    print("[PSEUDO] Skipping pseudo-label integration")

print('OK Data loaded')

In [ ]:
if DEBUG:
    EPOCHS = 1; FOLDS = [0]
    audio_cache_meta = audio_cache_meta.groupby('primary_label').head(3).reset_index(drop=True)
    sc_cache_meta = sc_cache_meta.head(50)
    Y_SC = Y_SC[:50]; non_s22_mask_sc = non_s22_mask_sc[:50]
    print(f'DEBUG: {len(audio_cache_meta)} focal, {len(sc_cache_meta)} sc')

## S3 — Model Architecture

In [ ]:
# =================================================================
# S3 -- EVAL UTILITIES + MEL + SED MODEL
# =================================================================

def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    if mask is not None: y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None: y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col): continue
        try: aucs.append(roc_auc_score(col, y_pred[:, c]))
        except: continue
    return (np.mean(aucs) if aucs else float('nan')), len(aucs)

def full_eval(y_true, y_pred, ns22, tm):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r['macro_auc_all'], r['n_all'] = round(a, 4), n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22)
    r['non_s22_macro'], r['n_ns22'] = round(a, 4), n
    for t, cm in tm.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22, class_mask=cm)
        r[f'non_s22_{t}'] = round(a, 4)
    return r

class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x):
        return self.db_transform(self.mel_spec(x))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)
    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS): mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS): mel = self.time_mask(mel)
        return mel

class PerchTeacher:
    def __init__(self, onnx_path):
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._embed_idx = next(
            (i for i, o in enumerate(self.session.get_outputs()) if o.shape and o.shape[-1] == PERCH_EMBED_DIM),
            1)
        print(f'Perch loaded: embed_idx={self._embed_idx}, providers={self.session.get_providers()}')
    @torch.no_grad()
    def embed(self, waveforms_5s):
        wav_np = waveforms_5s.cpu().numpy() if isinstance(waveforms_5s, torch.Tensor) else waveforms_5s
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))

def load_backbone_weights_offline(backbone):
    if not os.path.exists(B1_WEIGHTS_PATH):
        if IS_COLAB:
            print(f'  B1 weights not found. Downloading from HuggingFace ...')
            from huggingface_hub import hf_hub_download
            os.makedirs(str(Path(B1_WEIGHTS_PATH).parent), exist_ok=True)
            hf_hub_download(repo_id='timm/tf_efficientnet_b1.ns_jft_in1k',
                            filename='model.safetensors',
                            local_dir=str(Path(B1_WEIGHTS_PATH).parent))
            print(f'  Downloaded: {B1_WEIGHTS_PATH}')
        else:
            print(f'  WARNING: weights not found at {B1_WEIGHTS_PATH}')
            return
    from safetensors.torch import load_file as _st_load
    sd = _st_load(B1_WEIGHTS_PATH)
    for k in list(sd.keys()):
        if sd[k].ndim == 4 and sd[k].shape[1] == 3:
            bk = dict(backbone.named_parameters()).get(k)
            if bk is not None and bk.shape[1] == 1:
                sd[k] = sd[k].mean(dim=1, keepdim=True)
                print(f'  Adapted {k}: 3ch->1ch')
    missing, unexpected = backbone.load_state_dict(sd, strict=False)
    print(f'  Loaded B1 weights: {len(missing)} missing, {len(unexpected)} unexpected')

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)

class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool='', drop_path_rate=drop_path_rate)
        load_backbone_weights_offline(self.backbone)
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            feat = self.backbone(torch.randn(1, 1, N_MELS, n_tf))
            self.backbone_dim = feat.shape[1]
            print(f'Backbone feat: {tuple(feat.shape)}')
        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25), nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.att = nn.Conv1d(hidden_dim, num_classes, 1)
        self.cla = nn.Conv1d(hidden_dim, num_classes, 1)
        nn.init.xavier_uniform_(self.att.weight); self.att.bias.data.fill_(0.)
        nn.init.xavier_uniform_(self.cla.weight); self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = self.distill_head(h) if (return_distill and hasattr(self, 'distill_head')) else None
        h_cls = h.detach() if USE_PERCH_DISTILL else h
        h_cls = self.gem_freq(h_cls).permute(0, 2, 1)
        h_cls = self.dense(h_cls).permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill: return clip_logits, fw, distill_emb
        elif return_framewise: return clip_logits, fw
        elif return_distill:   return clip_logits, distill_emb
        return clip_logits

def make_model():
    return BirdSEDModel(BACKBONE_NAME).to(device)

print('OK Model definitions ready')

## S4 — Data Pipeline (embed_cache 対応)

In [ ]:
# =================================================================
# S4 -- DATA PIPELINE (torchaudio ベース)
# =================================================================

def extract_chunk_np(waveform, start_sample, n_samples):
    total = len(waveform)
    if total <= n_samples: return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total: start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w

# focal: cache_file = "label/filename"
def load_focal(cache_file):
    if cache_file in _AUDIO_CACHE: return _AUDIO_CACHE[cache_file]
    label, filename = cache_file.split('/', 1)
    path = resolve_focal_path(label, filename)
    if path is None: return None
    return load_audio_np(path)

# soundscape: filename のみ
_SC_WAV_CACHE = {}
def load_sc_waveform(sc_filename):
    if sc_filename in _SC_WAV_CACHE: return _SC_WAV_CACHE[sc_filename]
    a = load_audio_np(SC_AUDIO_DIR / sc_filename)
    if a is None: return None
    if len(_SC_WAV_CACHE) >= 30:
        _SC_WAV_CACHE.pop(next(iter(_SC_WAV_CACHE)))
    _SC_WAV_CACHE[sc_filename] = a
    return a

def _focal_embed(cache_file):
    key = f'focal/{cache_file}'
    emb = EMBED_CACHE.get(key)
    return torch.from_numpy(emb.copy()) if emb is not None else torch.zeros(PERCH_EMBED_DIM)

def _sc_embed(filename, start_sec):
    key = f'sc/{filename}/{start_sec}'
    emb = EMBED_CACHE.get(key)
    return torch.from_numpy(emb.copy()) if emb is not None else torch.zeros(PERCH_EMBED_DIM)

# SC MixUp pool
sc_mixup_sources = []
_labeled_rows = []
for i in range(len(sc_cache_meta)):
    row = sc_cache_meta.iloc[i]
    if Y_SC[i].sum() > 0:
        _labeled_rows.append({'filename': row['filename'], 'start_sec': int(row['start_sec']),
                               'label_idx': i, 'fold': int(row.get('fold', -1))})
if _labeled_rows:
    _labeled_meta = pd.DataFrame(_labeled_rows)
    sc_mixup_sources.append((_labeled_meta, Y_SC))
    print(f'SC MixUp pool: {len(_labeled_meta)} windows')


class FocalDS(Dataset):
    def __init__(self, df, l2i, secondary_lookup=None, sc_mixup_sources=None, fold_k=None, aug=False):
        self.df = df.reset_index(drop=True)
        self.l2i = l2i
        self.aug = aug
        self.secondary_lookup = secondary_lookup
        self.sc_mixup_sources = sc_mixup_sources
        self.fold_k = fold_k

    def __len__(self): return len(self.df)

    def _load_chunk(self, r):
        w = load_focal(r['cache_file'])
        if w is None: return None, None
        start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if self.aug and len(w) > TRAIN_SAMPLES else 0
        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r['primary_label']) in self.l2i:
            lb[self.l2i[str(r['primary_label'])]] = 1.0
        if self.secondary_lookup is not None and 'original_idx' in self.df.columns:
            for s in self.secondary_lookup.get(int(r['original_idx']), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return ch, lb

    def __getitem__(self, i):
        r1 = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r1)
        emb1 = _focal_embed(r1['cache_file'])

        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES),
                    'focal_missing', torch.zeros(PERCH_EMBED_DIM))

        if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            for _ in range(3):
                r2 = self.df.iloc[np.random.randint(len(self.df))]
                ch2, lb2 = self._load_chunk(r2)
                if ch2 is not None: break
            else: ch2 = None
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb2
                emb_mix = lam * emb1 + (1 - lam) * _focal_embed(r2['cache_file'])
                return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'focal', emb_mix)

        if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
                and np.random.random() < FOCAL_SC_MIXUP_PROB):
            meta_df_sc, labels = self.sc_mixup_sources[np.random.randint(len(self.sc_mixup_sources))]
            eligible = meta_df_sc[meta_df_sc['fold'] != self.fold_k] if self.fold_k is not None else meta_df_sc
            if len(eligible) > 0:
                sc_row = eligible.iloc[np.random.randint(len(eligible))]
                sc_wav = load_sc_waveform(sc_row['filename'])
                if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
                    sc_chunk = extract_chunk_np(sc_wav, int(sc_row['start_sec']) * SR, TRAIN_SAMPLES)
                    lam = np.random.beta(FOCAL_SC_MIXUP_ALPHA, FOCAL_SC_MIXUP_ALPHA)
                    ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
                    if self.aug: ch_mix = apply_aug(ch_mix)
                    lb_sc = labels[int(sc_row['label_idx'])].astype(np.float32)
                    lb = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
                    emb_mix = lam * emb1 + (1 - lam) * _sc_embed(sc_row['filename'], int(sc_row['start_sec']))
                    return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                            torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'focal', emb_mix)

        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0), torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'focal', emb1)


class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y = Y
        self.df = sc_df.reset_index(drop=True)
        self.aug = aug

    def __len__(self): return len(self.Y)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav_full = load_sc_waveform(row['filename'])
        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row['start_sec']) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
        emb = _sc_embed(row['filename'], int(row['start_sec']))
        return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), 'sc', emb)


class MixSamp(torch.utils.data.Sampler):
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs: per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]: self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                batch.extend([off + int(i) for i in self.rng.integers(0, size, size=n)])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch],
            torch.stack([b[5] for b in batch]))

def mk_sw(sr):
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

focal_secondary_labels = None
if USE_FOCAL_SECONDARY:
    focal_secondary_labels = {}
    for idx, row in train_df.iterrows():
        sec = row.get('secondary_labels', '')
        if pd.isna(sec) or sec in ('', '[]'): continue
        try: sec_list = eval(sec) if isinstance(sec, str) else []
        except: continue
        valid = [s for s in sec_list if s in LABEL2IDX]
        if valid: focal_secondary_labels[idx] = valid
    print(f'Secondary labels: {len(focal_secondary_labels)} files')

print('OK Data pipeline ready')

## S5 — Perch プリ計算 + 学習

In [ ]:
# =================================================================
# S5 -- PERCH PRECOMPUTE + TRAINING
# =================================================================

def precompute_all_embeddings(perch_teacher):
    global EMBED_CACHE
    EMBED_CACHE = {}
    EMBED_BATCH = 64
    t0 = time.time()

    # --- Focal ---
    unique_focal = audio_cache_meta.drop_duplicates('original_idx').reset_index(drop=True)
    print(f'[Precompute] focal: {len(unique_focal)} files (batch={EMBED_BATCH})')
    batch_wavs, batch_keys = [], []

    for i, (_, row) in enumerate(unique_focal.iterrows()):
        wav = load_focal(row['cache_file'])
        if wav is None: continue
        chunk = extract_chunk_np(wav, 0, TRAIN_SAMPLES).astype(np.float32)
        if len(chunk) < TRAIN_SAMPLES: chunk = np.pad(chunk, (0, TRAIN_SAMPLES - len(chunk)))
        batch_wavs.append(chunk)
        batch_keys.append(f"focal/{row['cache_file']}")

        if len(batch_wavs) >= EMBED_BATCH:
            embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
            for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e
            batch_wavs, batch_keys = [], []

        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            eta = (len(unique_focal) - i - 1) / ((i + 1) / elapsed)
            print(f'  focal {i+1}/{len(unique_focal)} | {elapsed:.0f}s | ETA {eta/60:.1f}min')

    if batch_wavs:
        embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
        for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e

    # --- SC windows ---
    print(f'[Precompute] SC: {len(sc_cache_meta)} windows')
    batch_wavs, batch_keys = [], []

    for i, row in sc_cache_meta.iterrows():
        wav = load_sc_waveform(row['filename'])
        if wav is None: continue
        chunk = extract_chunk_np(wav, int(row['start_sec']) * SR, TRAIN_SAMPLES).astype(np.float32)
        if len(chunk) < TRAIN_SAMPLES: chunk = np.pad(chunk, (0, TRAIN_SAMPLES - len(chunk)))
        batch_wavs.append(chunk)
        batch_keys.append(f"sc/{row['filename']}/{int(row['start_sec'])}")

        if len(batch_wavs) >= EMBED_BATCH:
            embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
            for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e
            batch_wavs, batch_keys = [], []

    if batch_wavs:
        embs = perch_teacher.embed(torch.from_numpy(np.stack(batch_wavs))).numpy()
        for k, e in zip(batch_keys, embs): EMBED_CACHE[k] = e

    elapsed = time.time() - t0
    print(f'[Precompute] Done: {len(EMBED_CACHE)} embeddings in {elapsed:.0f}s ({elapsed/60:.1f}min)')


def _load_val_waveforms(val_sc_df):
    wavs = []
    for _, row in val_sc_df.iterrows():
        w = load_sc_waveform(row['filename'])
        if w is not None:
            chunk = extract_chunk_np(w, int(row['start_sec']) * SR, VAL_SAMPLES)
            wavs.append(torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0))
        else:
            wavs.append(torch.zeros(1, VAL_SAMPLES))
    return wavs

def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            for i in range(mel.size(0)): mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
            p_clip = torch.sigmoid(clip_logits).cpu().numpy()
            p_fmax = torch.sigmoid(frame_max).cpu().numpy()
            preds_clip.append(p_clip); preds_fmax.append(p_fmax)
            preds_blend.append(0.5 * p_clip + 0.5 * p_fmax)
    return {'clip':  np.concatenate(preds_clip),
            'fmax':  np.concatenate(preds_fmax),
            'blend': np.concatenate(preds_blend)}

def build_active_datasets(fold_k):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta['fold'] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(('focal', fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta['fold'].values == fold_k
        sds = ScDS(Y_SC[~vm], sc_cache_meta[~vm].reset_index(drop=True), aug=True)
        items.append(('sc', sds, len(sds)))
    return items

LOG_INTERVAL = 50

def train_fold(fold_k):
    vm       = sc_cache_meta['fold'].values == fold_k
    Y_val    = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    val_sc_df = sc_cache_meta[vm].reset_index(drop=True)

    print(f'[fold {fold_k}] Loading Perch teacher ...')
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH)
    print(f'[fold {fold_k}] Pre-computing Perch embeddings ...')
    precompute_all_embeddings(perch_teacher)
    del perch_teacher; gc.collect()
    print(f'[fold {fold_k}] Perch teacher released. Training is now GPU-bound.')

    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))
    print(f'  Streams: {dict(zip(names, sizes))}  steps/ep: {nst}')

    m = make_model()
    mel_transform = MelSpecTransform().to(device)
    spec_augment  = SpecAugment().to(device)

    opt    = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = nst * WARMUP_EPOCHS
    total_steps  = nst * EPOCHS
    sch = torch.optim.lr_scheduler.SequentialLR(
        opt,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=1/25, end_factor=1.0, total_iters=warmup_steps),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup_steps, eta_min=1e-6),
        ],
        milestones=[warmup_steps])

    history = {'ep': [], 'train_loss': [], 'cls_loss': [], 'dist_loss': [],
               'ns22_macro': [], 'ns22_Aves': [], 'ns22_Amphibia': [],
               'ns22_Insecta': [], 'ns22_Mammalia': []}
    best_ns22, best_state_ns22 = -1.0, None
    best_macro, best_state_macro = -1.0, None
    val_wavs = _load_val_waveforms(val_sc_df)

    for ep in range(EPOCHS):
        m.train()
        smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        tl  = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m,
                         num_workers=4, pin_memory=True)
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        t0 = time.time()
        print(f'  Ep{ep:02d} start [{time.strftime("%H:%M:%S")}]')

        for step, (wav, lb, wt, mk, sr, perch_emb) in enumerate(tl):
            wav, lb, wt, mk = wav.to(device), lb.to(device), wt.to(device), mk.to(device)
            perch_emb = perch_emb.to(device)
            sw = mk_sw(sr).to(device)

            with torch.no_grad():
                mel = mel_transform(wav)
                for i in range(mel.size(0)): mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)

            with autocast():
                if USE_PERCH_DISTILL:
                    clip_logits, framewise, distill_emb = m(mel, return_framewise=True, return_distill=True)
                else:
                    clip_logits, framewise = m(mel, return_framewise=True)

                frame_max_logits = framewise.max(dim=1).values
                bce = 0.5 * F.binary_cross_entropy_with_logits(clip_logits, lb, reduction='none') \
                    + 0.5 * F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction='none')
                cls_loss = ((bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8) * sw).mean()

                if USE_PERCH_DISTILL:
                    distill_loss = F.mse_loss(distill_emb, perch_emb)
                    loss = cls_loss + ALPHA_DISTILL * distill_loss
                else:
                    distill_loss = torch.tensor(0.0)
                    loss = cls_loss

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()

            el += loss.item(); el_cls += cls_loss.item(); el_dist += distill_loss.item(); nb_count += 1

            if (step + 1) % LOG_INTERVAL == 0:
                elapsed = time.time() - t0
                sps = (step + 1) / elapsed
                eta = (nst - step - 1) / sps
                dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
                print(f"  Ep{ep:02d} [{step+1:4d}/{nst}] "
                      f"loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
                      f"lr={opt.param_groups[0]['lr']:.1e} "
                      f"| {sps:.2f}it/s ETA:{eta/60:.1f}min [{time.strftime('%H:%M:%S')}]")

        val_preds_dict = _predict_from_waveforms(m, mel_transform, val_wavs)
        val_preds = val_preds_dict['blend']
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        for mode in ['clip', 'fmax', 'blend']:
            r[f'ns22_{mode}'] = full_eval(Y_val, val_preds_dict[mode], ns22_val, TAXON_MASKS)['non_s22_macro']

        history['ep'].append(ep)
        history['train_loss'].append(round(el / nb_count, 5))
        history['cls_loss'].append(round(el_cls / nb_count, 5))
        history['dist_loss'].append(round(el_dist / nb_count, 5))
        history['ns22_macro'].append(r['non_s22_macro'])
        for t in ['Aves', 'Amphibia', 'Insecta', 'Mammalia']:
            history[f'ns22_{t}'].append(r[f'non_s22_{t}'])

        tag_ns22 = tag_macro = ''
        if r['non_s22_macro'] > best_ns22:
            best_ns22 = r['non_s22_macro']
            best_state_ns22 = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_ns22 = ' *ns22'
        if r['macro_auc_all'] > best_macro:
            best_macro = r['macro_auc_all']
            best_state_macro = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_macro = ' *macro'

        dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
        ep_time  = time.time() - t0
        print(f"    Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
              f"| ns22: {r['ns22_blend']:.4f} "
              f"Av={r['non_s22_Aves']:.4f} Am={r['non_s22_Amphibia']:.4f} "
              f"In={r['non_s22_Insecta']:.4f} Ma={r['non_s22_Mammalia']:.4f} "
              f"[{ep_time/60:.1f}min]{tag_ns22}{tag_macro}")

    del m, mel_transform, spec_augment
    torch.cuda.empty_cache(); gc.collect()
    return best_state_ns22, best_state_macro, history

print('OK Training function ready')

## S6 — Fold Loop + ONNX Export

In [ ]:
# =================================================================
# S6 -- FOLD LOOP + ONNX EXPORT
# =================================================================

if MODE != 'train':
    print("Skipping training (MODE='infer')")
    oof_ns22 = None; all_hist = {}
else:
    oof_ns22 = np.full((len(sc_cache_meta), NUM_CLASSES), np.nan, dtype=np.float32)
    all_hist = {}

    for fold_k in FOLDS:
        print(f"\n{'='*60}\nFOLD {fold_k}\n{'='*60}")
        vm = sc_cache_meta['fold'].values == fold_k
        val_sc_df_k = sc_cache_meta[vm].reset_index(drop=True)
        print(f'[S6] val windows: {vm.sum()}')

        best_ns22_state, best_macro_state, hist = train_fold(fold_k)
        print(f'[S6] train_fold done. best_macro={"set" if best_macro_state else "None"}')
        all_hist[fold_k] = hist

        mel_tf = MelSpecTransform().to(device)
        val_wavs_k = _load_val_waveforms(val_sc_df_k)

        if best_macro_state is not None:
            ckpt_path = OUT_DIR / f'fold{fold_k}_best_macro.pt'
            print(f'[S6] Saving checkpoint -> {ckpt_path}')
            torch.save(best_macro_state, ckpt_path)

            print('[S6] OOF prediction ...')
            m = make_model()
            m.load_state_dict(best_macro_state, strict=False)
            oof_ns22[vm] = _predict_from_waveforms(m, mel_tf, val_wavs_k)['blend']
            print(f'[S6] OOF done: shape={oof_ns22[vm].shape}')

            # ONNX export
            m.eval()
            INF_N_MELS   = 128
            INF_N_FRAMES = VAL_SAMPLES // HOP_LENGTH + 1

            class SEDExportWrapper(nn.Module):
                def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
                    super().__init__()
                    self.backbone = timm.create_model(backbone_name, pretrained=False, in_chans=1,
                                                      num_classes=0, global_pool='', drop_path_rate=0.1)
                    self.gem_freq    = GeMFreqPool(p_init=3.0)
                    self.dense_drop1 = nn.Dropout(0.25)
                    self.dense_conv  = nn.Conv1d(backbone_dim, hidden_dim, 1)
                    self.dense_relu  = nn.ReLU(inplace=True)
                    self.dense_drop2 = nn.Dropout(0.5)
                    self.att         = nn.Conv1d(hidden_dim, num_classes, 1)
                    self.cla         = nn.Conv1d(hidden_dim, num_classes, 1)
                def forward(self, mel):
                    h = self.backbone(mel)
                    h = self.gem_freq(h)
                    h = self.dense_drop1(h)
                    h = self.dense_conv(h)
                    h = self.dense_relu(h)
                    h = self.dense_drop2(h)
                    norm_att  = torch.softmax(torch.tanh(self.att(h)), dim=-1)
                    framewise = self.cla(h)
                    clip      = torch.sum(norm_att * framewise, dim=2)
                    return clip, framewise.permute(0, 2, 1)

            def load_and_remap_state(export_model, trained_state):
                remap = {}
                for k, v in trained_state.items():
                    if k.startswith('distill_head.'): continue
                    if k == 'dense.1.weight': remap['dense_conv.weight'] = v.unsqueeze(-1)
                    elif k == 'dense.1.bias': remap['dense_conv.bias'] = v
                    else: remap[k] = v
                export_model.load_state_dict(remap, strict=False)

            print(f'[S6] Building export model (backbone_dim={m.backbone_dim}) ...')
            export_model = SEDExportWrapper(BACKBONE_NAME, NUM_CLASSES, m.backbone_dim).to(device)
            load_and_remap_state(export_model, best_macro_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f'sed_distill_fold{fold_k}.onnx'
            print(f'[S6] Exporting ONNX -> {onnx_path} ...')
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=['mel'], output_names=['clip_logits', 'framewise_logits'],
                dynamic_axes={'mel': {0: 'batch'}, 'clip_logits': {0: 'batch'},
                              'framewise_logits': {0: 'batch'}},
                opset_version=18)

            # 単一ファイルに統合（.onnx.data が別ファイルになるのを防ぐ）
            import onnx as _onnx
            _model = _onnx.load(str(onnx_path))
            _onnx.save(_model, str(onnx_path), save_as_external_data=False)
            _data_path = onnx_path.with_suffix('.onnx.data')
            if _data_path.exists():
                _data_path.unlink()
                print(f'[S6] Merged external data into single file')

            print('[S6] Verifying ONNX ...')
            _sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
            _out  = _sess.run(None, {'mel': dummy_mel.cpu().numpy()})
            with torch.no_grad(): _ref, _ = export_model(dummy_mel)
            _diff = np.abs(_ref.cpu().numpy() - _out[0]).max()
            print(f'  ONNX verify: max|diff|={_diff:.3e}')
            assert _diff < 0.01, f'ONNX diverged: {_diff}'
            del _sess

            size_mb = onnx_path.stat().st_size / 1e6
            print(f'  Exported {onnx_path.name} ({size_mb:.1f} MB)')
            del m, export_model
            print(f'[S6] Fold {fold_k} complete!')
        else:
            print(f'[S6] WARNING: best_macro_state is None for fold {fold_k}')

## S7 — OOF Evaluation

In [ ]:
# =================================================================
# S7b -- FAIR OOF RE-EVALUATION (real labels only)
# =================================================================
# Filter to chunks where is_pseudo_mask == False (i.e., real labels in train_soundscapes_labels.csv)
# This removes the circular pseudo-vs-pseudo evaluation bias
# =================================================================
from sklearn.metrics import roc_auc_score
import numpy as np

if oof_ns22 is not None:
    # Filter: real labels + non-S22 + has predictions
    real_mask = ~is_pseudo_mask
    has_pred = ~np.isnan(oof_ns22[:, 0])
    fair_mask = real_mask & non_s22_mask_sc & has_pred

    print(f"Total chunks: {len(Y_SC)}")
    print(f"  Real labels (~is_pseudo): {real_mask.sum()}")
    print(f"  non-S22:                  {non_s22_mask_sc.sum()}")
    print(f"  With predictions (val):   {has_pred.sum()}")
    print(f"  FAIR eval set:            {fair_mask.sum()}")

    if fair_mask.sum() == 0:
        print("\n⚠️  No chunks meet fair eval criteria. Cannot evaluate.")
    else:
        y_true = Y_SC[fair_mask]
        y_pred = oof_ns22[fair_mask]

        # Per-class AUC (only classes with positives)
        aucs_per_class = {}
        for c in range(NUM_CLASSES):
            if y_true[:, c].sum() > 0:
                try:
                    aucs_per_class[c] = roc_auc_score(y_true[:, c], y_pred[:, c])
                except ValueError:
                    pass

        macro_auc = np.mean(list(aucs_per_class.values())) if aucs_per_class else float('nan')
        print(f"\n{'='*60}")
        print(f"FAIR OOF RESULTS (real labels only)")
        print(f"{'='*60}")
        print(f"  macro AUC:        {macro_auc:.4f}")
        print(f"  classes evaluated: {len(aucs_per_class)} / {NUM_CLASSES}")

        # By taxon
        for taxon, idx_arr in TAXON_MASKS.items():
            taxon_aucs = [aucs_per_class[c] for c in idx_arr if c in aucs_per_class]
            if taxon_aucs:
                print(f"    {taxon:12s}: {np.mean(taxon_aucs):.4f}  ({len(taxon_aucs)} classes)")
            else:
                print(f"    {taxon:12s}: nan  (0 classes)")

        print(f"\n📊 Baseline comparison:")
        print(f"  Original B1 fold 0     (hard labels):  0.7148")
        print(f"  Original B1 fold 3+4   (hard labels):  0.8282")
        print(f"  New B1 fold 0/3/4 with pseudo (fair):  {macro_auc:.4f}")

        # Per-fold breakdown
        print(f"\n📈 Per-fold (fair) AUC:")
        for fold_k in FOLDS:
            fold_mask = (sc_cache_meta['fold'].values == fold_k) & fair_mask
            if fold_mask.sum() == 0:
                print(f"  fold {fold_k}: no eligible chunks")
                continue
            yt = Y_SC[fold_mask]; yp = oof_ns22[fold_mask]
            fold_aucs = []
            for c in range(NUM_CLASSES):
                if yt[:, c].sum() > 0:
                    try: fold_aucs.append(roc_auc_score(yt[:, c], yp[:, c]))
                    except ValueError: pass
            print(f"  fold {fold_k} ({fold_mask.sum()} chunks): macro AUC = {np.mean(fold_aucs):.4f} ({len(fold_aucs)} classes)")
else:
    print("oof_ns22 is None (MODE != 'train'). Run S6 first.")

In [ ]:
# =================================================================
# S7 -- OOF EVALUATION
# =================================================================
if MODE == 'train' and oof_ns22 is not None:
    has = ~np.isnan(oof_ns22[:, 0])
    if has.sum() > 0:
        r_all = full_eval(Y_SC[has], oof_ns22[has], non_s22_mask_sc[has], TAXON_MASKS)
        print('=' * 60)
        print('OOF RESULTS')
        print('=' * 60)
        print(f"  macro AUC (all):      {r_all['macro_auc_all']:.4f}")
        print(f"  macro AUC (non-S22):  {r_all['non_s22_macro']:.4f}")
        for t in ['Aves', 'Amphibia', 'Insecta', 'Mammalia']:
            print(f"    {t:<12}: {r_all.get(f'non_s22_{t}', float('nan')):.4f}")

In [ ]:
# =================================================================
# S8 -- ONNX ダウンロード (Colab のみ)
# 学習完了後にこのセルを実行して ONNX ファイルをローカルに保存
# .onnx と .onnx.data の両方をダウンロードします
# =================================================================
if IS_COLAB:
    from google.colab import files
    import glob

    # .onnx と .onnx.data を両方取得
    onnx_files = sorted(glob.glob('/content/working/*.onnx'))
    data_files  = sorted(glob.glob('/content/working/*.onnx.data'))
    all_files   = onnx_files + data_files

    if all_files:
        print(f'Downloading {len(onnx_files)} .onnx + {len(data_files)} .onnx.data files:')
        for f in all_files:
            size_mb = os.path.getsize(f) / 1e6
            print(f'  {os.path.basename(f)}  ({size_mb:.1f} MB)')
            files.download(f)
        print('Done!')
    else:
        print('No ONNX files found in /content/working/')
        print('Available files:', os.listdir('/content/working/'))